# A fully spectral kinematic-match model: guided tour

This notebook walks through `SpectralKM` end to end: what the model represents, how to run
one impact, how to read the diagnostics, and where the model is known to be weak. It is
meant to be read alongside `docs/next-gen-KM-model.tex`, which is the physics ground truth;
section references below point into it.

**No plotting dependency.** Figures are built as SVG strings and handed to Jupyter directly,
so this notebook runs against the package's own `Project.toml` with nothing extra installed.

**The model in one paragraph.** A droplet and a bath are each represented by a truncated
spectral expansion: the bath surface as $\eta=\sum_{m}a_mJ_0(k_mr)$ on Fourier--Bessel modes
of a finite container of radius $b$, the droplet as $\xi=1+\sum_l\beta_lP_l(\cos\theta)$ on
Legendre modes. They interact only through a contact pressure $p$, expanded on shifted
Legendre polynomials over the *moving* contact patch $\theta\in[0,\theta_c]$. Each mode
responds to pressure through a variable-step BDF2 discretisation that makes every state
variable **affine** in its own pressure moment, so one timestep reduces to a small square
nonlinear system in the pressure coefficients alone. The contact angle $\theta_c$ is *not*
an unknown of that system: it is selected outside it as the feasibility boundary
$\theta_c=\inf\{\theta:\text{non-penetration holds}\}$. That nesting is what makes the inner
problem well conditioned --- condition number $O(1)$ and flat in the timestep, against
$\sim10^{18}$ for the joint system.

There are no collocation points anywhere: every contact condition is imposed weakly, as a
Galerkin projection against the pressure basis in a self-adjoint pairing.

In [ ]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))
using SpectralKM, Printf, SpecialFunctions

## 1. Parameters, and how the truncations are chosen

`Params` carries the physics (`We`, `Bo`, `Oh`, bath radius `b`, depth `h0`) and the
numerics (`M` bath modes, `L` droplet modes, `N` pressure modes, `nq` quadrature nodes).

The truncations are not interchangeable, and the reasoning differs for each:

| | meaning | how to choose |
|---|---|---|
| `M` | bath Fourier--Bessel modes | resolve the shortest capillary wave that carries energy; $M=60$ at $b=6$ |
| `L` | droplet Legendre modes | resolve the contact patch: need $L\gtrsim\pi/\theta_c$ near onset |
| `N` | pressure modes on the patch | **small on purpose** --- see below |
| `nq` | Gauss nodes on the patch | exact for the polynomial degrees involved; $nq\ge2(N+L)$ |

`N` deserves the caveat. The compliance operator mapping pressure to gap displacement is
*compact*, so its singular values decay and the pointwise pressure profile **never
converges** in `N` --- only its low-order moments, which is all the dynamics ever sees.
Raising `N` therefore buys resolution the model cannot use and costs conditioning. `N=3` is
the production value not because it is converged pointwise but because the integrated
quantities are insensitive to it, which cell 6 checks directly.

In [ ]:
p = Params(We=1.0958, Bo=0.017, Oh=0.006, M=60, L=60, N=3, b=6.0, h0=3.0, nq=40)
@printf("We=%.4f  Bo=%.4f  Oh=%.4f   M=%d L=%d N=%d nq=%d\n", p.We, p.Bo, p.Oh, p.M, p.L, p.N, p.nq)
@printf("wall condition: %s\n", p.wall)
@printf("first five bath wavenumbers k_m: %s\n", join(round.(p.k[1:5], digits=4), ", "))
println()
println("k_0 = 0 is the piston (uniform) mode; it exists only for a free wall.")
println("Its pressure response is exactly zero, so it cannot be driven -- volume is conserved.")

## 2. One impact

`run_simulation` returns three index-aligned things: `levels` (the state at every accepted
step), `diag` (one row per **contact** step, carrying $\theta_c$, the force $f$, and the
solver's own diagnostics), and `phases` (a label per level). Note `diag` is shorter than
`levels` and indexes differently --- that is why `phases` exists.

In [ ]:
levels, diag, phases = run_simulation(p; t_end=14.0, dt_init=1e-3)
times = [lv.t for lv in levels]
@printf("accepted steps: %d   of which in contact: %d   t_final: %.3f\n",
        length(levels), length(diag), times[end])
@printf("impact speed: %.4f   rebound speed: %.4f\n", -sqrt(p.We), levels[end].com.v)

## 3. Contact time: three definitions, one correct

The single most important trap in reading this model's output. A single physical impact does
**not** produce a single contact interval: the stepper detaches and immediately re-attaches
a few times, because the `just_left` guard forces one advancing free-flight step after
contact ends, after which onset is re-detected at the next step. Those re-entries are
separated by a gap of exactly `dt_init`, which is their signature.

So three scalars all answer to "the contact time", and they differ by about 17%:

- `primary_contact_time` --- the first interval's duration. **This is the metric.** It is the
  only one that measures a single physical event, and the one to converge in `M`, `L`, `N`.
- `contact_time` --- the sum over all intervals. Includes the chatter. Legitimate for a
  dissipation budget (total time under load), wrong for an impact duration.
- the first-to-last **span** --- includes chatter *and* any free-flight excursion between
  re-contacts. Never use it. Earlier revisions of this project reported it by mistake.

Run the cell and read the interval table: the physical contact is the long first row, and
everything after it is a handful of steps.

In [ ]:
ivs = contact_intervals(times, phases)
@printf("%-4s %-10s %-10s %-10s %-7s\n", "#", "t_start", "t_end", "duration", "steps")
for (i, iv) in enumerate(ivs)
    @printf("%-4d %-10.4f %-10.4f %-10.4f %-7d\n", i, iv.t_start, iv.t_end, iv.duration, iv.nsteps)
end
for k in 2:length(ivs)
    @printf("  gap before #%d: %.4f%s\n", k, ivs[k].t_start - ivs[k-1].t_end,
            isapprox(ivs[k].t_start - ivs[k-1].t_end, 1e-3; rtol=0.2) ? "   <- one dt_init: chatter" : "")
end
println()
@printf("primary_contact_time : %.4f   <- the metric\n", primary_contact_time(times, phases))
@printf("contact_time (total) : %.4f\n", contact_time(times, phases))
@printf("span (do not use)    : %.4f\n", ivs[end].t_end - ivs[1].t_start)
@printf("\ncoefficient of restitution: %.4f\n", coefficient_of_restitution(times, levels, phases))
@printf("max penetration depth     : %.4f\n", max_penetration_depth(levels, p.L))

## 4. A minimal SVG plotter

Enough to draw polylines on axes, with no dependency. Single-quoted XML attributes keep the
Julia strings free of escapes.

In [ ]:
function svgplot(series; width=680, height=300, xlabel="", ylabel="", title="",
                 xlim=nothing, ylim=nothing)
    pad = 52
    xs = vcat((s.x for s in series)...); ys = vcat((s.y for s in series)...)
    x0, x1 = xlim === nothing ? (minimum(xs), maximum(xs)) : xlim
    y0, y1 = ylim === nothing ? (minimum(ys), maximum(ys)) : ylim
    x1 == x0 && (x1 = x0 + 1); y1 == y0 && (y1 = y0 + 1)
    sx = v -> pad + (v - x0) / (x1 - x0) * (width - 1.6pad)
    sy = v -> height - pad - (v - y0) / (y1 - y0) * (height - 1.7pad)
    io = IOBuffer()
    print(io, "<svg xmlns='http://www.w3.org/2000/svg' width='$width' height='$height' font-family='sans-serif' font-size='12'>")
    print(io, "<rect width='$width' height='$height' fill='white'/>")
    # axes with ticks
    for (frac) in 0:0.25:1
        xv = x0 + frac * (x1 - x0); yv = y0 + frac * (y1 - y0)
        print(io, "<line x1='$(sx(xv))' y1='$(height-pad)' x2='$(sx(xv))' y2='$(height-pad+5)' stroke='black'/>")
        print(io, "<text x='$(sx(xv))' y='$(height-pad+18)' text-anchor='middle'>$(round(xv,sigdigits=3))</text>")
        print(io, "<line x1='$pad' y1='$(sy(yv))' x2='$(pad-5)' y2='$(sy(yv))' stroke='black'/>")
        print(io, "<text x='$(pad-9)' y='$(sy(yv)+4)' text-anchor='end'>$(round(yv,sigdigits=3))</text>")
    end
    print(io, "<line x1='$pad' y1='$(height-pad)' x2='$(width-0.6pad)' y2='$(height-pad)' stroke='black'/>")
    print(io, "<line x1='$pad' y1='$(height-pad)' x2='$pad' y2='$(pad*0.5)' stroke='black'/>")
    for s in series
        pts = join(("$(sx(s.x[i])),$(sy(s.y[i]))" for i in eachindex(s.x)), " ")
        w = get(s, :width, 1.8)
        print(io, "<polyline points='$pts' fill='none' stroke='$(s.color)' stroke-width='$w'/>")
    end
    for (i, s) in enumerate(series)
        haskey(s, :label) || continue
        print(io, "<line x1='$(width-190)' y1='$(pad*0.5+14i)' x2='$(width-165)' y2='$(pad*0.5+14i)' stroke='$(s.color)' stroke-width='2.4'/>")
        print(io, "<text x='$(width-159)' y='$(pad*0.5+14i+4)'>$(s.label)</text>")
    end
    print(io, "<text x='$(width/2)' y='$(height-8)' text-anchor='middle'>$xlabel</text>")
    print(io, "<text x='14' y='$(height/2)' text-anchor='middle' transform='rotate(-90 14 $(height/2))'>$ylabel</text>")
    print(io, "<text x='$(width/2)' y='18' text-anchor='middle' font-size='14'>$title</text></svg>")
    return HTML(String(take!(io)))
end

struct HTML s::String end
Base.show(io::IO, ::MIME"text/html", h::HTML) = print(io, h.s)
println("ready")

## 5. The trajectory, and the contact diagnostics

The droplet's south pole against the bath deflection under it, then the two quantities the
closure actually produces: the selected contact angle $\theta_c(\tau)$ and the net force
$f(\tau)=2\int p\,w\,dx$.

In [ ]:
south = [lv.com.z - xi_of_theta(lv.drop.beta, 0.0, p.L) for lv in levels]
eta0  = [sum(lv.bath.a[m+1] for m in 0:p.M) for lv in levels]   # eta(r=0), since J_0(0)=1
svgplot([(x=times, y=south, color="#1f77b4", label="droplet south pole"),
         (x=times, y=eta0,  color="#d62728", label="bath surface at r=0")];
        xlabel="tau", ylabel="height", title="Impact trajectory")

In [ ]:
td = [d.t for d in diag]
svgplot([(x=td, y=[d.theta_c for d in diag], color="#2ca02c", label="theta_c")];
        xlabel="tau", ylabel="theta_c", title="Contact angle: selected, not solved for")

In [ ]:
svgplot([(x=td, y=[d.f for d in diag], color="#9467bd", label="f"),
         (x=[first(td), last(td)], y=[0.0, 0.0], color="#bbbbbb", width=1.0)];
        xlabel="tau", ylabel="f", title="Net contact force")

Two things to notice in the force trace. It is positive almost throughout --- the pressure
pushes the droplet *up*, as it must --- and it can dip transiently negative early without
contact having ended. That dip is why loss of contact requires $f\le0$ on **three
consecutive** steps rather than a single sign change: an eager test sends the stepper back
to free flight, where onset is immediately re-detected and an expensive onset search is paid
again. At $We=0.3$ that cost a twentyfold slowdown before the streak condition was added.

## 6. Interface shapes, and what the pressure does not do

Reconstruct both interfaces at a few instants during contact.

In [ ]:
contact_idx = findall(==(InContact), phases)
picks = contact_idx[round.(Int, range(1, length(contact_idx); length=4))]
rg = range(0.0, 3.0; length=240)
cols = ["#08306b", "#2171b5", "#6baed6", "#c6dbef"]
series = NamedTuple[]
for (c, i) in zip(cols, picks)
    push!(series, (x=collect(rg), y=reconstruct_bath(levels[i], collect(rg), p),
                   color=c, label="tau=$(round(times[i],digits=2))"))
end
svgplot(series; xlabel="r", ylabel="eta", title="Bath surface during contact")

In [ ]:
# Sensitivity of the INTEGRATED metrics to the pressure truncation N.
# The pointwise pressure profile does not converge in N (the compliance operator is
# compact); the question is whether that matters for what we report. Runs a few impacts.
@printf("%-4s %-12s %-12s %-10s\n", "N", "t_cont", "CoR", "r_c max")
for N in (2, 3, 6)
    pn = Params(We=1.0958, Bo=0.017, Oh=0.006, M=60, L=60, N=N, b=6.0, h0=3.0, nq=max(40, 10N))
    lv, dg, ph = run_simulation(pn; t_end=14.0, dt_init=1e-3)
    tt = [l.t for l in lv]
    @printf("%-4d %-12.5f %-12.5f %-10.4f\n", N, primary_contact_time(tt, ph),
            coefficient_of_restitution(tt, lv, ph), sin(maximum(d.theta_c for d in dg)))
end

## 7. Wall condition: free versus pinned contact line

Two container configurations, selected by `wall`. See design doc §subsubsec:wall.

- `:free` (default) --- no-flux walls and a free $90^\circ$ contact line, $\partial_r\eta=0$
  at $r=b$. Basis: zeros of $J_1$, plus the piston mode $k_0=0$. Weight $2/(bJ_0(k_mb))^2$.
- `:pinned` --- the surface is pinned at the triple point, $\eta(b)=0$, wall slope free.
  Basis: zeros of $J_0$, **no** piston mode. Weight $2/(bJ_1(k_mb))^2$.

The pinned option buys exact pinning at a real price, stated plainly: it violates no-flux at
the wall **at leading order**, since $\partial_r\phi(b)\propto J_1(k_mb)$ which is $O(1)$ for
this basis. It is offered because that failure is one quantified broken boundary condition,
whereas the alternative (no-flux basis plus a pinning constraint) cannot represent a pinned
meniscus with a nonzero wall slope at all. Neither has been validated against a pinned-bath
experiment, because no such dataset is available here.

In [ ]:
for wall in (:free, :pinned)
    pw = Params(We=1.0958, Bo=0.017, Oh=0.006, M=60, L=60, N=3, b=6.0, h0=3.0, nq=40, wall=wall)
    lv, dg, ph = run_simulation(pw; t_end=14.0, dt_init=1e-3)
    tt = [l.t for l in lv]
    # eta at the wall, worst case over the whole run
    wallmax = maximum(abs(sum(l.bath.a[m+1] * besselj0(pw.k[m+1] * pw.b) for m in 0:pw.M)) for l in lv)
    @printf("%-8s t_cont=%.4f  CoR=%.4f  max|eta(b)| over run = %.2e\n",
            wall, primary_contact_time(tt, ph), coefficient_of_restitution(tt, lv, ph), wallmax)
end
println()
println("For :pinned, |eta(b)| sits at roundoff throughout -- pinning is an identity of the")
println("basis, not a constraint being enforced, so it cannot drift.")

## 8. Sweeps, and validation

For many cases use `scripts/sweep.jl`, which runs cases concurrently at a worker count it
picks by measuring rather than guessing:

```
julia --project=. -t auto scripts/sweep.jl --wall=free 0.2 0.4 1.0958 3.0
```

It calibrates one case's marginal memory footprint (about 12 MiB, against a fixed ~450 MiB
runtime cost), caps the worker count by a memory budget, then time-ablates $W=1,2,4,\dots$
and takes the knee. On an 8-core machine the knee is typically $W=4$: going to 8 workers buys
about 6%, because these runs contend on memory bandwidth and the GC. The sweep is resumable
--- completed Weber numbers are skipped on a re-run.

**Validation is against experiment only.** `scripts/validate_experimental.jl` compares the
droplet top and bottom trajectories against measured data with digitised error bars. The
model sits at about 1.4x the experimental error-bar half-length. Scripts that compare
against the 1PKM model are named `compare_*`, not `validate_*`, deliberately: another
model's output is not evidence.

**Known open issues**, none hidden:

- The contact radius peaks at $\tau\approx0.89$ against DNS's $1.46$. Not a resolution
  effect; refining `N` moves it *away* from DNS, implicating the unresolved pressure profile.
  No experimental contact radius exists to adjudicate.
- Convergence in `M`, `L` and the spectral cutoff is **not** established. Only `N`
  insensitivity of the integrated metrics is checked (cell 6).
- The detachment chatter of cell 3 is a stepper artifact that has not been eliminated, only
  measured and routed around by reporting the primary interval.